In [8]:
# ============================================
# 1. Import Libraries
# ============================================
import sys
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    f1_score,
)

# ============================================
# 2. Configure Project Path
# ============================================
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ============================================
# 3. Import Project Modules
# ============================================
from src.config import (
    TARGET,
    TIER1_FEATURES,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
)

from src.preprocessing import (
    replace_false_with_nan,
    convert_numeric_columns,
)

from src.evaluation import (
    evaluate_model,
    cross_validate_model,
    plot_confusion_matrix,
    plot_roc,
    plot_precision_recall,
)

# ============================================
# 4. Load Dataset
# ============================================
df = pd.read_csv(
    PROJECT_ROOT / "data" / "SEED-ML" / "infertility_man_data-v2.csv",
    sep=";",
)

/var/folders/qy/pmwwp9z11l9gx48kp84g7q640000gn/T/ipykernel_72806/2873492366.py:55: DtypeWarning: Columns (0: sample_vol_initial, 1: sample_concentration_initial, 2: sample_production_total, 3: sample_num_prog_mob_total, 4: sample_num_spz_normal_pre, 5: sample_morphology_normal_pre, 6: sample_num_spz_normal_post, 7: sample_morphology_normal_post) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [2]:
# ============================================
# 5. Define Preprocessing Pipeline
# ============================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ]
)

In [5]:
# ============================================
# 6. Prepare Modeling Dataset
# ============================================

X = df[TIER1_FEATURES].copy()
y = df[TARGET].copy()

# Replace encoded missing values
X_clean = replace_false_with_nan(X)

# Convert numeric features to numeric dtype
X_clean = convert_numeric_columns(
    X_clean,
    NUMERIC_FEATURES,
)

In [6]:
# ============================================
# 7. Train-Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

print("\nTraining class distribution (%)")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTest class distribution (%)")
print((y_test.value_counts(normalize=True) * 100).round(2))

Training set: (8099, 18)
Test set: (2025, 18)

Training class distribution (%)
diagnostic
NO                               62.69
OLIGOASTHENOTHERATOZOOSPERMIA    14.22
ASTHENOZOOSPERMIA                11.66
THERATOZOOSPERMIA                 6.70
OLIGOZOOSPERMIA                   1.89
ASTHENOTHERATOZOOSPERMIA          1.38
OLIGOASTHENOZOOSPERMIA            0.96
OLIGOTHERATOZOOSPERMIA            0.33
AZOOSPERMIA                       0.16
Name: proportion, dtype: float64

Test class distribution (%)
diagnostic
NO                               62.67
OLIGOASTHENOTHERATOZOOSPERMIA    14.22
ASTHENOZOOSPERMIA                11.65
THERATOZOOSPERMIA                 6.72
OLIGOZOOSPERMIA                   1.93
ASTHENOTHERATOZOOSPERMIA          1.38
OLIGOASTHENOZOOSPERMIA            0.94
OLIGOTHERATOZOOSPERMIA            0.35
AZOOSPERMIA                       0.15
Name: proportion, dtype: float64


In [7]:
# ============================================
# 8. Train Logistic Regression Model
# ============================================

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

model.fit(
    X_train,
    y_train,
)

y_pred = model.predict(X_test)

In [9]:
# ============================================
# 9. Evaluate Model
# ============================================

print(classification_report(
    y_test,
    y_pred,
))

balanced_accuracy = balanced_accuracy_score(
    y_test,
    y_pred,
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
)

print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

                               precision    recall  f1-score   support

     ASTHENOTHERATOZOOSPERMIA       0.54      0.68      0.60        28
            ASTHENOZOOSPERMIA       0.98      0.80      0.88       236
                  AZOOSPERMIA       0.40      0.67      0.50         3
                           NO       0.95      1.00      0.98      1269
OLIGOASTHENOTHERATOZOOSPERMIA       0.96      0.70      0.81       288
       OLIGOASTHENOZOOSPERMIA       0.75      0.79      0.77        19
       OLIGOTHERATOZOOSPERMIA       0.12      0.86      0.21         7
              OLIGOZOOSPERMIA       0.39      0.62      0.48        39
            THERATOZOOSPERMIA       0.87      0.76      0.81       136

                     accuracy                           0.90      2025
                    macro avg       0.66      0.76      0.67      2025
                 weighted avg       0.93      0.90      0.91      2025

Balanced Accuracy: 0.7634
Macro F1: 0.6709


In [10]:
# ============================================
# 9. Evaluate Model
# ============================================

test_results = evaluate_model(
    model,
    X_test,
    y_test,
)

test_results

Classification Report
                               precision    recall  f1-score   support

     ASTHENOTHERATOZOOSPERMIA       0.54      0.68      0.60        28
            ASTHENOZOOSPERMIA       0.98      0.80      0.88       236
                  AZOOSPERMIA       0.40      0.67      0.50         3
                           NO       0.95      1.00      0.98      1269
OLIGOASTHENOTHERATOZOOSPERMIA       0.96      0.70      0.81       288
       OLIGOASTHENOZOOSPERMIA       0.75      0.79      0.77        19
       OLIGOTHERATOZOOSPERMIA       0.12      0.86      0.21         7
              OLIGOZOOSPERMIA       0.39      0.62      0.48        39
            THERATOZOOSPERMIA       0.87      0.76      0.81       136

                     accuracy                           0.90      2025
                    macro avg       0.66      0.76      0.67      2025
                 weighted avg       0.93      0.90      0.91      2025

Balanced Accuracy: 0.7634
Macro F1:          0.6709


{'balanced_accuracy': 0.7634121398469801,
 'macro_f1': 0.6708808633460568,
 'weighted_f1': 0.9102743379105007}